# ChartGemma on the protected 100-question holdout

Runs a chart-QA specialist model (`ahmed-masry/chartgemma`, ~3B) on the exact same 100-question holdout Yahor's six systems were benchmarked on, using CharXiv's own vendored generation code unmodified. Tests whether a model *specialized* for chart QA reads charts better than the general-purpose Qwen2.5-VL-3B this whole project is built around -- a different lever than anything tried so far (training/alignment on a fixed student, or judge quality). Scored with this project's own whole-token matcher for direct comparability with every other number here.

No secrets needed: repo, model weights, and holdout images are all public and already present after cloning.


In [ ]:
!git clone https://github.com/yahorlahunovich/chart-prm.git
%cd chart-prm
# chart_prm.text_match (this session's work) only exists on the ertugrul branch,
# not yet merged to main -- plain clone checks out main by default.
!git checkout ertugrul
# No -U, no pillow/torchvision named: this project already learned the hard way
# (implementation_log.md, "PyTorch Binary Preservation") that upgrading pillow/torch
# on Kaggle's preinstalled image breaks torchvision's C extension / import chain.
# transformers alone gets a version floor (same fix pattern the Qwen notebooks already
# needed) since PaliGemma support requires a newer release than Kaggle ships by default;
# transformers does not hard-depend on pillow, so this shouldn't touch it.
!pip install -q "transformers>=4.49.0" accelerate sentencepiece

# The 100 holdout images were never committed to git (only the 500 training-pool ones
# were) -- download just this subset, same script the rest of the project uses.
!python scripts/data_prep/download_images.py --ids-file data/splits/eval_reasoning_ids.json


In [ ]:
import json
import sys
sys.path.append("data/CharXiv/src")
sys.path.append("src")

from reasoning_utils import build_reasoning_queries
from generate_lib.chartgemma import generate_response
from chart_prm.text_match import answers_match

with open("data/CharXiv/data/reasoning_val.json", "r") as f:
    all_data = json.load(f)

# The same 100-question holdout every other system in this project was benchmarked on
holdout_ids = set(["1857", "646", "1386", "1533", "1863", "587", "1696", "2224", "1474", "813", "2082", "1489", "1895", "991", "662", "1544", "1600", "1773", "814", "1634", "523", "1953", "2330", "1469", "1306", "1536", "1026", "1974", "1503", "223", "1999", "1246", "379", "193", "252", "2313", "777", "1842", "1918", "872", "2067", "1182", "198", "1401", "1437", "1619", "409", "147", "638", "1320", "177", "1060", "236", "1130", "1124", "2050", "2193", "451", "1630", "530", "513", "2198", "1043", "894", "905", "1686", "1886", "1262", "1726", "1261", "1007", "154", "86", "167", "1722", "1039", "1010", "2136", "1818", "83", "1117", "1774", "349", "339", "1903", "2245", "516", "1521", "144", "537", "2399", "547", "1618", "1387", "1229", "671", "1472", "1259", "1307", "1490"])
assert len(holdout_ids) == 100

# figure_id is stored as an int in reasoning_val.json; holdout_ids (from
# data/splits/eval_reasoning_ids.json) are strings, matching how the rest of this
# project treats question IDs -- compare as strings on both sides.
holdout_data = {k: d for k, d in all_data.items() if str(d["figure_id"]) in holdout_ids}
print(f"{len(holdout_data)} holdout questions found in reasoning_val.json (expect 100)")
assert len(holdout_data) == 100

queries = build_reasoning_queries(holdout_data, image_dir="data/CharXiv/images")
print(f"{len(queries)} queries built")


In [ ]:
# CharXiv's own ChartGemma adapter, completely unmodified -- mutates `queries` in
# place, adding a 'response' key to each entry.
generate_response(queries, "ahmed-masry/chartgemma")
print("Generation complete.")


In [ ]:
ground_truth_by_id = {d["figure_id"]: d["answer"] for d in holdout_data.values()}

rows = []
n_correct = 0
for figure_id, q in queries.items():
    gt = ground_truth_by_id[figure_id]
    pred = q.get("response", "")
    correct = answers_match(gt, pred)
    n_correct += int(correct)
    rows.append({
        "question_id": figure_id,
        "raw_question": q["raw_question"],
        "ground_truth": gt,
        "response": pred,
        "correct": correct,
    })

accuracy = n_correct / len(rows)
print(f"ChartGemma accuracy on the 100-question holdout: {n_correct}/{len(rows)} = {accuracy:.1%}")

with open("chartgemma_holdout_results.jsonl", "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
with open("chartgemma_holdout_accuracy.json", "w", encoding="utf-8") as f:
    json.dump({"n_correct": n_correct, "n_total": len(rows), "accuracy": accuracy}, f, indent=2)
print("Saved chartgemma_holdout_results.jsonl and chartgemma_holdout_accuracy.json")
